In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ------------------------------------------
# 1. Expanded Dataset
# ------------------------------------------
# Using < and > as start/end tokens for clarity
data = [
    ("hello", "नमस्ते"),
    ("how are you", "आप कैसे हैं"),
    ("i am fine", "मैं ठीक हूँ"),
    ("thank you", "धन्यवाद"),
    ("good night", "शुभ रात्रि"),
    ("good morning", "शुभ प्रभात"),
    ("welcome", "स्वागत है"),
    ("see you", "फिर मिलते हैं"),
    ("what is your name", "आपका नाम क्या है"),
    ("my name is gemini", "मेरा नाम जेमिनी है"),
    ("i am happy", "मैं खुश हूँ"),
    ("excuse me", "क्षमा करें"),
    ("yes", "हाँ"),
    ("no", "नहीं"),
    ("please", "कृपया")
]

input_texts = [pair[0] for pair in data]
# Add start (<) and end (>) tokens to target
target_texts = ['<' + pair[1] + '>' for pair in data]

# ------------------------------------------
# 2. Tokenization & Preprocessing
# ------------------------------------------
# Encoder Tokenizer
in_tok = Tokenizer(char_level=True)
in_tok.fit_on_texts(input_texts)
enc_seq = in_tok.texts_to_sequences(input_texts)

# Decoder Tokenizer (filters='' to keep < and >)
out_tok = Tokenizer(char_level=True, filters='')
out_tok.fit_on_texts(target_texts)
dec_seq = out_tok.texts_to_sequences(target_texts)

max_enc_len = max(len(s) for s in enc_seq)
max_dec_len = max(len(s) for s in dec_seq)

# Pad sequences
encoder_input_data = pad_sequences(enc_seq, maxlen=max_enc_len, padding='post')
decoder_input_data = pad_sequences(dec_seq, maxlen=max_dec_len, padding='post')

# Target data: Same as decoder_input but shifted by one character
decoder_target_data = np.zeros((len(data), max_dec_len, 1), dtype="float32")
for i, seq in enumerate(dec_seq):
    for t, char_idx in enumerate(seq):
        if t > 0:
            # The target for the decoder at time t is the character at t+1
            decoder_target_data[i, t - 1, 0] = char_idx

# Vocab sizes
num_enc_tokens = len(in_tok.word_index) + 1
num_dec_tokens = len(out_tok.word_index) + 1
latent_dim = 128 # Increased memory capacity

# ------------------------------------------
# 3. Model Architecture (Training)
# ------------------------------------------
# Encoder
enc_inputs = Input(shape=(max_enc_len,))
enc_emb = Embedding(num_enc_tokens, latent_dim)(enc_inputs)
_, enc_state = SimpleRNN(latent_dim, return_state=True)(enc_emb)

# Decoder
dec_inputs = Input(shape=(max_dec_len,))
dec_emb_layer = Embedding(num_dec_tokens, latent_dim)
dec_emb = dec_emb_layer(dec_inputs)
dec_rnn = SimpleRNN(latent_dim, return_sequences=True, return_state=True)
dec_outputs, _ = dec_rnn(dec_emb, initial_state=enc_state)
dec_dense = Dense(num_dec_tokens, activation='softmax')
dec_outputs = dec_dense(dec_outputs)

model = Model([enc_inputs, dec_inputs], dec_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# ------------------------------------------
# 4. Training
# ------------------------------------------
print("Training model... please wait.")
model.fit([encoder_input_data, decoder_input_data], decoder_target_data, 
          epochs=600, verbose=0)
print("Training Complete!")

# ------------------------------------------
# 5. Inference Setup (The Logic for Prediction)
# ------------------------------------------
encoder_model = Model(enc_inputs, enc_state)

inf_state_input = Input(shape=(latent_dim,))
inf_dec_inputs = Input(shape=(1,))
inf_dec_emb = dec_emb_layer(inf_dec_inputs)
inf_dec_out, inf_dec_state = dec_rnn(inf_dec_emb, initial_state=inf_state_input)
inf_dec_out = dec_dense(inf_dec_out)

decoder_model = Model([inf_dec_inputs, inf_state_input], [inf_dec_out, inf_dec_state])

rev_out_char = {v: k for k, v in out_tok.word_index.items()}

def translate(input_word):
    # Prepare input
    test_seq = in_tok.texts_to_sequences([input_word.lower()])
    test_seq = pad_sequences(test_seq, maxlen=max_enc_len, padding='post')
    
    # Get initial state from encoder
    states_value = encoder_model.predict(test_seq, verbose=0)
    
    # Start character (<)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = out_tok.word_index['<']
    
    decoded_sentence = ""
    stop_condition = False
    
    while not stop_condition:
        output_tokens, h = decoder_model.predict([target_seq, states_value], verbose=0)
        
        # Pick the character with highest probability
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = rev_out_char.get(sampled_token_index, "")
        
        # Exit if end token or max length reached
        if sampled_char == '>' or len(decoded_sentence) > max_dec_len:
            stop_condition = True
        else:
            decoded_sentence += sampled_char
            
        # Update for next step
        target_seq[0, 0] = sampled_token_index
        states_value = h
        
    return decoded_sentence

# ------------------------------------------
# 6. Interactive Testing
# ------------------------------------------
print("\n--- Translation Test ---")
print("Type a sentence from the dataset (e.g., 'how are you')")
while True:
    txt = input("\nEnter English (or 'exit'): ")
    if txt.lower() == 'exit': break
    try:
        print(f"Hindi: {translate(txt)}")
    except KeyError:
        print("Error: Word contains characters the model hasn't seen.")

Training model... please wait.


C:\Users\User\anaconda3\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor_15', 'keras_tensor_19']. Received: the structure of inputs=('*', '*')
  warnings.warn(


Training Complete!

--- Translation Test ---
Type a sentence from the dataset (e.g., 'how are you')

Enter English (or 'exit'): hello


C:\Users\User\anaconda3\Lib\site-packages\keras\src\models\functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['keras_tensor_25', 'keras_tensor_24']. Received: the structure of inputs=('*', '*')
  warnings.warn(


Hindi: नमस्ते

Enter English (or 'exit'): how are you
Hindi: आप कैसे हैं

Enter English (or 'exit'): excuse me
Hindi: क्षमा करें

Enter English (or 'exit'): bye
Hindi: हाँ

Enter English (or 'exit'): see you
Hindi: फिर मिलते हैं

Enter English (or 'exit'): exit
